In [1]:
print('running')

import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import openai

test = pd.read_csv("test.csv")
checkpoint_file = "llm_results.csv"

# load already processed indices if file exists
done_indices = set()
if os.path.exists(checkpoint_file):
    prev = pd.read_csv(checkpoint_file)
    if "i" in prev.columns:
        done_indices = set(prev["i"].tolist())
        print(f"Resuming: {len(done_indices)} rows already done")

start = time.time()

class_sci_api_key = "sk-j49dRg-v03_qWy-1xijenA"

prompt = "You are an expert human vs AI classifier and will determine the source of a text. Output exactly a 0 for human or 1 for AI with no extra explanation or delimiters. Here is the text you will classify\n"

# lock to protect file writes
write_lock = threading.Lock()

def call_llm(prompt_text, model="gemma3"):
    client = openai.OpenAI(
        api_key=class_sci_api_key,
        base_url="https://ol.sci.pitt.edu"
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt_text}]
    )
    return response.choices[0].message.content

def process_row(i, row):
    text = row["text"]
    label = row["label"]

    try:
        response = call_llm(prompt + text, model="gemma3")
        pred_text = str(response).strip()

        if pred_text not in {"0", "1"}:
            pred_text = pred_text[0] if pred_text and pred_text[0] in {"0", "1"} else None

        prediction = int(pred_text) if pred_text is not None else 0

    except Exception as e:
        print(f"iteration {i} failed: {e}")
        prediction = 0   # default to 0 on failure

    row_dict = {
        "i": i,
        "text": text,
        "label": label,
        "prediction": prediction
    }

    return i, row_dict

# rows still to do
remaining = [(i, row) for i, row in test.iterrows() if i not in done_indices]

print(f"Remaining rows: {len(remaining)}")

max_workers = 20

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_row, i, row) for i, row in remaining]

    for future in as_completed(futures):
        i, row_dict = future.result()

        print(f"iteration {i} output: {row_dict['prediction']} label: {row_dict['label']}")

        with write_lock:
            pd.DataFrame([row_dict]).to_csv(
                checkpoint_file,
                mode="a",
                header=not os.path.exists(checkpoint_file),
                index=False
            )

end = time.time()
duration = round(end - start, 2)
print(f"duration: {duration}s")

running
ERROR! Session/line number was not unique in database. History logging moved to new session 116
Resuming: 77547 rows already done
Remaining rows: 7352
iteration 77555 output: 1 label: 0
iteration 77556 output: 0 label: 1
iteration 77553 output: 0 label: 0
iteration 77547 output: 1 label: 1
iteration 77550 output: 1 label: 1
iteration 77554 output: 1 label: 1
iteration 77546 output: 0 label: 1
iteration 77561 output: 1 label: 1
iteration 77552 output: 1 label: 1
iteration 77560 output: 0 label: 0
iteration 77558 output: 1 label: 1
iteration 77563 output: 0 label: 0
iteration 77566 output: 0 label: 0
iteration 77559 output: 1 label: 0
iteration 77557 output: 1 label: 1
iteration 77565 output: 0 label: 1
iteration 77562 output: 0 label: 1
iteration 77564 output: 0 label: 1
iteration 77548 output: 1 label: 1
iteration 77549 output: 1 label: 0
iteration 77567 output: 1 label: 1
iteration 77574 output: 0 label: 1
iteration 77568 output: 1 label: 1
iteration 77571 output: 1 label: 1
i